# 01 — Data Collection & Understanding
## FMD Outbreak Prediction — Sri Lanka (2017–2024)

**Objective:** This notebook introduces all the raw data sources used in this
research project. Each dataset is loaded and explored so we can understand
its structure, coverage, and quality before any processing.

### Data Sources
| # | Source | Description | Resolution |
|---|--------|-------------|------------|
| 1 | DAPH | FMD outbreak records from Department of Animal Production & Health | Monthly, by district |
| 2 | CHIRPS | Rainfall estimates from Climate Hazards Group | Dekadal → monthly, by district |
| 3 | GLW | Gridded Livestock of the World (buffalo & cattle density) | Spatial grid, 2015 & 2020 |
| 4 | Admin Boundaries | Sri Lanka district shapefiles (OCHA/HDX) | District level (admin2) |
| 5 | Weather | Temperature, humidity, wind speed (external) | Monthly, by district |

---
## 1. FMD Outbreak Records (DAPH)

The Department of Animal Production & Health (DAPH) of Sri Lanka maintains
records of Foot-and-Mouth Disease outbreaks across all 25 districts.
This is our **target variable** — whether an outbreak occurred in a given
district-month.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

BASE_DIR = r'..'
RAW_DIR  = os.path.join(BASE_DIR, 'data', 'raw')
PLOT_DIR = os.path.join(BASE_DIR, 'plots', '01_data_overview')
os.makedirs(PLOT_DIR, exist_ok=True)

# Load DAPH outbreak data
daph_file = os.path.join(RAW_DIR, 'daph', 'FMD_SriLanka_2017_2024 (without case counts) - usable.xlsx')
daph = pd.read_excel(daph_file)

print(f"Shape: {daph.shape}")
print(f"Columns: {daph.columns.tolist()}")
print(f"\nData types:\n{daph.dtypes}")
print(f"\nFirst 5 rows:")
daph.head()

Shape: (2400, 4)
Columns: ['Year', 'District', 'Month', 'Outbreak status']

Data types:
Year               int64
District             str
Month                str
Outbreak status    int64
dtype: object

First 5 rows:


,Year,District,Month,Outbreak status
0,2017,Ampara,January,0
1,2017,Ampara,February,0
2,2017,Ampara,March,0
3,2017,Ampara,April,0
4,2017,Ampara,May,0


### Key observations — DAPH data
- **2400 rows** = 25 districts × 8 years (2017–2024) × 12 months
- **Target column:** `Outbreak status` (0 = no outbreak, 1 = outbreak)
- No missing values expected — this is a complete monthly panel

In [13]:
# Basic statistics
print(f"Year range: {daph['Year'].min()} – {daph['Year'].max()}")
print(f"Districts: {daph['District'].nunique()} → {sorted(daph['District'].unique())}")
print(f"\nOutbreak status distribution:")
print(daph['Outbreak status'].value_counts())
print(f"\nMissing values:\n{daph.isnull().sum()}")

Year range: 2017 – 2024
Districts: 25 → ['Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Jaffna', 'Kalutara', 'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar', 'Matale', 'Matara', 'Moneragala', 'Mullaitivu', 'Nuwara Eliya', 'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya']

Outbreak status distribution:
Outbreak status
0    2172
1     228
Name: count, dtype: int64

Missing values:
Year               0
District           0
Month              0
Outbreak status    0
dtype: int64


---
## 2. Rainfall Data (CHIRPS)

CHIRPS (Climate Hazards Group InfraRed Precipitation with Station data)
provides high-resolution rainfall estimates. The data is at **dekadal**
(10-day) resolution and will be aggregated to monthly during preprocessing.

Key rainfall features:
- `rfh` — Total rainfall (mm) for the dekad
- `r1h` — 1-day rainfall maximum
- `r3h` — 3-day rainfall maximum
- `rfq` — Rainfall frequency (number of rainy days)

In [14]:
# Load CHIRPS rainfall data
chirps_file = os.path.join(RAW_DIR, 'chirps', 'Rainfall data from 1981 - 2026.csv')
chirps = pd.read_csv(chirps_file)

print(f"Shape: {chirps.shape}")
print(f"Columns: {chirps.columns.tolist()}")
print(f"\nData types:\n{chirps.dtypes}")
print(f"\nFirst 5 rows:")
chirps.head()

Shape: (55488, 15)
Columns: ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version']

Data types:
date             str
adm_level      int64
adm_id         int64
PCODE            str
n_pixels       int64
rfh          float64
rfh_avg      float64
r1h          float64
r1h_avg      float64
r3h          float64
r3h_avg      float64
rfq          float64
r1q          float64
r3q          float64
version          str
dtype: object

First 5 rows:


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,1/1/1981,2,25830,LK22,67,105.970146,71.977610,NaN,NaN,NaN,NaN,144.159000,NaN,NaN,final
1,1/11/1981,2,25830,LK22,67,7.716418,56.896020,NaN,NaN,NaN,NaN,20.544806,NaN,NaN,final
2,1/21/1981,2,25830,LK22,67,4.328358,54.214924,118.014920,183.08856,NaN,NaN,15.753390,65.402664,NaN,final
3,2/1/1981,2,25830,LK22,67,54.358208,44.043285,66.402985,155.15424,NaN,NaN,121.032290,44.583885,NaN,final
4,2/11/1981,2,25830,LK22,67,3.731343,30.813930,62.417908,129.07214,NaN,NaN,24.379740,50.284798,NaN,final


In [15]:
# Filter to our study period (2017–2024)
chirps['date'] = pd.to_datetime(chirps['date'], format='mixed', dayfirst=True)
chirps_study = chirps[(chirps['date'].dt.year >= 2017) & (chirps['date'].dt.year <= 2024)]
print(f"\nFiltered to 2017–2024: {chirps_study.shape[0]} rows")
print(f"Date range: {chirps_study['date'].min()} to {chirps_study['date'].max()}")
print(f"Unique PCODEs: {chirps_study['PCODE'].nunique()}")
print(f"\nBasic stats for rainfall (rfh):")
print(chirps_study['rfh'].describe())


Filtered to 2017–2024: 9792 rows
Date range: 2017-01-01 00:00:00 to 2024-12-21 00:00:00
Unique PCODEs: 34

Basic stats for rainfall (rfh):
count    9792.000000
mean       61.888920
std        63.793473
min         0.000000
25%        13.818726
50%        41.354663
75%        89.184224
max       532.574100
Name: rfh, dtype: float64


---
## 3. Livestock Density (GLW)

The Gridded Livestock of the World (GLW) dataset provides spatial
density estimates for different livestock species. We use **buffalo**
and **cattle** density data as they are the primary hosts for FMD.

Data is available for **2015** and **2020** reference years.

In [16]:
# Check what GLW data files are available
glw_dir = os.path.join(RAW_DIR, 'glw')
for animal in ['buffolo', 'cattle']:
    animal_dir = os.path.join(glw_dir, animal)
    for year_dir in os.listdir(animal_dir):
        year_path = os.path.join(animal_dir, year_dir)
        if os.path.isdir(year_path):
            files = os.listdir(year_path)
            print(f"\n{animal}/{year_dir}: {len(files)} files")
            for f in files[:5]:
                print(f"  - {f}")
            if len(files) > 5:
                print(f"  ... and {len(files)-5} more")


buffolo/2015: 2 files
  - area_km2.tif
  - buffalo_2015.tif

buffolo/2020: 1 files
  - buffalo_2020.tif

cattle/2015: 2 files
  - area_km2.tif
  - cattle_2015.tif

cattle/2020: 1 files
  - cattle_2020.tif


> **Note:** GLW data is in raster (GeoTIFF) format. During preprocessing
> (Notebook 02), we extract district-level mean density values by overlaying
> the raster with Sri Lanka's admin boundary shapefiles.

---
## 4. Administrative Boundaries

Sri Lanka's district boundaries (shapefiles) from OCHA/HDX are used to:
- Map PCODE identifiers to district names
- Extract zonal statistics from raster datasets (GLW, weather)
- Generate spatial visualizations

In [17]:
# Check available shapefiles
admin_dir = os.path.join(RAW_DIR, 'lka_admin_boundaries')
shp_files = [f for f in os.listdir(admin_dir) if f.endswith('.shp')]
print("Available shapefiles:")
for f in shp_files:
    print(f"  - {f}")

# Try loading district-level boundaries (admin2 = district level in Sri Lanka)
try:
    import geopandas as gpd
    shp_path = os.path.join(admin_dir, 'lka_admin2.shp')
    districts = gpd.read_file(shp_path)
    print(f"\nDistrict boundaries loaded: {len(districts)} features")
    print(f"Columns: {districts.columns.tolist()}")

    # Find likely name and pcode columns (handle different source conventions)
    name_col = None
    pcode_col = None
    for col in districts.columns:
        cl = col.lower()
        if name_col is None and any(k in cl for k in ['adm2_en', 'adm2_name', 'name', 'district', 'adm2']) :
            name_col = col
        if pcode_col is None and any(k in cl for k in ['pcode', 'pcodes', 'adm2_pcode', 'code']):
            pcode_col = col

    # Explicit fallbacks (in case of uppercase columns)
    if name_col is None and 'ADM2_EN' in districts.columns:
        name_col = 'ADM2_EN'
    if pcode_col is None and 'ADM2_PCODE' in districts.columns:
        pcode_col = 'ADM2_PCODE'

    if name_col and pcode_col:
        print(f"Using columns: {name_col}, {pcode_col}")
        display_cols = [name_col, pcode_col]
        print(districts[display_cols].head(10))
    else:
        print("Could not find standard name/pcode columns. Showing first 5 rows for inspection:")
        print(districts.head())

except ImportError:
    print("\n⚠️ geopandas not installed. Install with: pip install geopandas")
    print("   Shapefiles will be used in Notebook 02 for spatial processing.")
except Exception as e:
    print(f"\nError loading shapefile or selecting columns: {e}")
    print("Inspect 'districts.columns' above to choose the correct columns manually.")

Available shapefiles:
  - lka_admin0.shp
  - lka_admin1.shp
  - lka_admin2.shp
  - lka_admin3.shp
  - lka_adminlines.shp
  - lka_adminpoints.shp

District boundaries loaded: 25 features
Columns: ['adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'center_lat', 'center_lon', 'geometry']
Using columns: adm2_name, adm2_pcode
      adm2_name adm2_pcode
0       Colombo       LK11
1       Gampaha       LK12
2      Kalutara       LK13
3         Kandy       LK21
4        Matale       LK22
5  Nuwara Eliya       LK23
6         Galle       LK31
7        Matara       LK32
8    Hambantota       LK33
9        Jaffna       LK41


---
## 5. Weather Data (Temperature, Humidity, Wind Speed)

Monthly weather variables (temperature, humidity, wind speed) were collected
per district. These are already incorporated into the processed intermediate
files. The original source data was obtained from meteorological records.

In [18]:
# Check intermediate processed files to verify weather data presence
proc_dir = os.path.join(BASE_DIR, 'data', 'processed', 'while Processing')
proc_files = os.listdir(proc_dir)
print("Intermediate processed files:")
for f in proc_files:
    fpath = os.path.join(proc_dir, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  - {f} ({size_mb:.2f} MB)")

# Load the merged intermediate file to check weather columns
merged = pd.read_csv(os.path.join(proc_dir, 'merged_final.csv'))
weather_cols = ['temperature', 'humidity', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1']
available = [c for c in weather_cols if c in merged.columns]
print(f"\nWeather columns in merged data: {available}")
print(f"\nWeather stats:")
print(merged[available].describe())

Intermediate processed files:
  - FMD_SriLanka_2017_2024_updated_with_date.csv (0.08 MB)
  - FMD_SriLanka_2017_2024_with_date_and_pcode.csv (0.09 MB)
  - merged_final.csv (0.63 MB)
  - merged_fmd_rainfall, monsoon_and_daph_final.xlsx (0.29 MB)
  - merged_fmd_rainfall_and_daph.csv.xlsx (0.27 MB)
  - merged_fmd_rainfall_and_daphold.csv.xlsx (0.27 MB)
  - merged_fmd_rain_humid_wind_temp_to_daph.csv (0.50 MB)

Weather columns in merged data: ['temperature', 'humidity', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1']

Weather stats:
       temperature     humidity   wind_speed    temp_lag1  humidity_lag1  \
count  2400.000000  2400.000000  2400.000000  2400.000000    2400.000000   
mean     26.684633    81.555712     3.422192    26.679389      81.552157   
std       1.855017     5.488415     1.298719     1.858642       5.490288   
min      20.510000    62.720000     0.960000    20.510000      62.720000   
25%      25.540000    77.600000     2.420000    25.540000      77.600000   
5

---
## Summary

| Data Source | Records | Key Columns | Status |
|-------------|---------|-------------|--------|
| DAPH (Outbreaks) | 2,400 | Year, District, Month, Outbreak status | ✅ Ready |
| CHIRPS (Rainfall) | ~100K+ | date, PCODE, rfh, r1h, r3h, rfq | ✅ Ready (needs monthly agg) |
| GLW (Livestock) | Raster | Buffalo & cattle density grids | ✅ Ready (needs zonal stats) |
| Admin Boundaries | 25 districts | District names, PCODEs, geometry | ✅ Ready |
| Weather | 2,400 | temperature, humidity, wind_speed | ✅ Already in processed files |

**Next:** Notebook 02 will merge all these sources into a single model-ready dataset.